In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
 
# 1. Build the training set from the data itself
df["theme"] = df["complaint_category"].fillna("Positive / No Complaint")
 
train = df[df["comment"].notna() & (df["comment"].str.strip() != "")]
train = train.drop_duplicates(subset=["comment"])[["comment", "theme"]]
 
# Since there are only 18 canned comments in the data, tech more to increase accuracy
# 2. Seed phrases: teach the model vocabulary the canned comments lack
seed = pd.DataFrame([
    ("I was billed twice and no one can explain the charge.", "Billing"),
    ("My insurance claim was denied and the invoice is wrong.", "Billing"),
    ("I'm still waiting for a refund on an overcharge.", "Billing"),
    ("The doctor was rude and dismissed my concerns.", "Provider"),
    ("The physician rushed me and barely listened.", "Provider"),
    ("The nurse was condescending during my exam.", "Provider"),
    ("The waiting room was dirty and cramped.", "Facilities"),
    ("The building was hard to navigate and parking was awful.", "Facilities"),
    ("The restroom was out of order and the clinic felt run down.", "Facilities"),
    ("I waited two hours past my scheduled appointment time.", "Wait Time"),
    ("It took months to get an appointment.", "Wait Time"),
    ("Nobody returned my calls about my test results.", "Communication"),
    ("I never received my lab results or follow-up instructions.", "Communication"),
    ("No one called me back after I left several messages.", "Communication"),
    ("Everyone was lovely, great visit!", "Positive / No Complaint"),
    ("Fantastic care, thank you so much.", "Positive / No Complaint"),
    ("Smooth visit, no complaints at all.", "Positive / No Complaint"),
], columns=["comment", "theme"])
 
train = pd.concat([train, seed], ignore_index=True)
 
# 3. Train the classifier
model_theme = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), MultinomialNB())
model_theme.fit(train["comment"], train["theme"])
 
# 4. Sanity check on a new, unseen comment
new_comment = "The waiting room was packed and I felt claustrophobic."
predicted_theme = model_theme.predict([new_comment])[0]
print(f"The AI classified '{new_comment}' as: {predicted_theme}")
 
overall = df["at_risk"].mean()
print(f"{len(df):,} responses | overall At-Risk rate: {overall:.1%}")
 
# ---------- 1. RATING DISTRIBUTION ----------
# ~38.6% of responses are At-Risk: a widespread issue, not a small unhappy group
counts = df["rating"].value_counts().sort_index()
colors = ["#D8492F" if r <= 3 else "#0F5C5C" for r in counts.index]
counts.plot(kind="bar", color=colors, figsize=(8, 4))
plt.title(f"Rating distribution — {overall:.1%} of responses are At Risk (coral)")
plt.xlabel("Rating"); plt.ylabel("Responses"); plt.xticks(rotation=0)
plt.show()
 
# ---------- 2. AT-RISK RATE BY THEME ----------
# Billing/Provider/Facilities convert to At-Risk 100% of the time
themes = df.groupby("theme").agg(count=("theme", "size"), risk=("at_risk", "mean"))
neg = themes[themes["risk"] > 0].sort_values("risk")
(neg["risk"] * 100).plot(kind="barh", color="#D8492F", figsize=(8, 4))
plt.axvline(overall * 100, color="gray", linestyle="--", label="org average")
plt.title("Share of each complaint theme classified At Risk")
plt.xlabel("At-Risk rate (%)"); plt.legend()
plt.show()
 
# ---------- 3. VOLUME VS SEVERITY ----------
# Wait Time is most common but not most damaging; Facilities is rare but always At-Risk
plt.figure(figsize=(8, 5))
plt.scatter(neg["count"], neg["risk"] * 100, s=neg["count"] / 50, color="#D8492F", alpha=0.7)
for name, row in neg.iterrows():
    plt.annotate(name, (row["count"], row["risk"] * 100 + 2), ha="center")
plt.title("Volume vs severity — frequent complaints are not the most damaging")
plt.xlabel("Number of complaints"); plt.ylabel("At-Risk rate (%)")
plt.show()
 
# ---------- 4. DEPARTMENT HOTSPOTS ----------
# 20.5% to 57.9% across departments (~3x): driven by practices, not inevitable
dept = df.groupby("department_id").agg(count=("rating", "size"), risk=("at_risk", "mean"))
dept = dept[dept["count"] >= 50].sort_values("risk", ascending=False)
(dept["risk"] * 100).plot(kind="bar", color="#0F5C5C", figsize=(12, 4))
plt.axhline(overall * 100, color="gray", linestyle="--", label="org average")
plt.title(f"At-Risk rate by department ({dept['risk'].min():.1%} to {dept['risk'].max():.1%})")
plt.ylabel("At-Risk rate (%)"); plt.xticks(fontsize=6); plt.legend()
plt.show()
 
# ---------- 5. QUARTERLY TREND ----------
# Flat at ~38-39% for 3 years: a persistent structural problem, not a recent spike
df["feedback_date"] = pd.to_datetime(df["feedback_date"], errors="coerce")
dated = df.dropna(subset=["feedback_date"])
trend = dated.groupby(dated["feedback_date"].dt.to_period("Q"))["at_risk"].mean()
(trend * 100).plot(marker="o", color="#D8492F", figsize=(9, 4))
plt.ylim(30, 45)
plt.title("Quarterly At-Risk rate — flat for 3 years (structural, not a spike)")
plt.ylabel("At-Risk rate (%)")
plt.show()
 
# ---------- 6. BOOKING DELAY VS AT-RISK ----------
# Extra: dissatisfaction climbs with scheduling lag (~20% within 3 days vs ~54% past 30)
df["wait_bucket"] = pd.cut(df["wait_days"], [0, 3, 7, 14, 21, 30, 70],
                           labels=["0-3", "4-7", "8-14", "15-21", "22-30", "31+"])
wait = df.groupby("wait_bucket", observed=True)["at_risk"].mean()
(wait * 100).plot(kind="bar", color="#B8862F", figsize=(8, 4))
plt.axhline(overall * 100, color="gray", linestyle="--", label="org average")
plt.title("At-Risk rate rises with booking delay")
plt.xlabel("Days between scheduling and appointment"); plt.ylabel("At-Risk rate (%)")
plt.xticks(rotation=0); plt.legend()
plt.show()
 
# ---------- 7. FUTURE NO-SHOWS ----------
# Extra: At-Risk patients no-show ~6x more often — the business cost of doing nothing
noshow = df.groupby("at_risk")["future_no_show_flag"].mean()
(noshow * 100).rename({False: "Satisfied", True: "At Risk"}).plot(
    kind="bar", color=["#0F5C5C", "#D8492F"], figsize=(6, 4))
plt.title(f"At-Risk patients no-show {noshow[True]/noshow[False]:.1f}x more often")
plt.ylabel("Future no-show rate (%)"); plt.xticks(rotation=0)
plt.show()
 
 
# ---------- 8. BOOKING DELAY BY RATING LEVEL ----------
# The lower the rating, the longer that patient waited to be seen:
# rating-1 patients averaged ~37 days vs ~16 days for rating-5 patients
delay = df.groupby("rating")["wait_days"].mean()
colors = ["#D8492F" if r <= 3 else "#0F5C5C" for r in delay.index]
delay.plot(kind="bar", color=colors, figsize=(8, 4))
plt.title("Average booking delay by rating — unhappy patients waited 2x longer")
plt.xlabel("Patient rating"); plt.ylabel("Avg days from scheduling to appointment")
plt.xticks(rotation=0)
plt.show()
 
# ---------- 9. THE COST OF NO-SHOWS ----------
# Price a no-show using MedCore's own billing data: the average
# completed visit generates ~$617 in claims. Every no-show forfeits that.
avg_revenue = df.loc[df["has_claim"] & (df["total_claim_amount"] > 0),
                     "total_claim_amount"].mean()
n_noshows = (df["status"] == "No Show").sum()
total_cost = n_noshows * avg_revenue
 
print(f"Average revenue per visit:  ${avg_revenue:,.0f}")
print(f"No-shows in dataset:        {n_noshows:,}")
print(f"Estimated lost revenue:     ${total_cost/1e6:.1f}M over 3 years "
      f"(~${total_cost/3/1e6:.1f}M per year)")
 
pd.Series({"Revenue kept\n(completed visits)": (df["status"] == "Completed").sum() * avg_revenue / 1e6,
           "Revenue lost\n(no-shows)": total_cost / 1e6}).plot(
    kind="bar", color=["#0F5C5C", "#D8492F"], figsize=(6, 4))
plt.title(f"No-shows cost MedCore ≈ ${total_cost/1e6:.0f}M in lost visit revenue")
plt.ylabel("Estimated revenue ($M)"); plt.xticks(rotation=0)
plt.show()
 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
 
# 1. Load the data
df = pd.read_csv("patient_experience_master.csv")
 
# 2. Clean the data
# Keep only valid ratings 1-5 and ensure no-show flag is clean
df = df[df["rating"].isin([1, 2, 3, 4, 5])].copy()
df = df.dropna(subset=["future_no_show_flag"])
df["future_no_show_flag"] = df["future_no_show_flag"].astype(int)
df["wait_days"] = df["wait_days"].fillna(df["wait_days"].median())
 
# 3. Feature Engineering: One-Hot Encoding for visit types
visit_type_dummies = pd.get_dummies(df["visit_type"], prefix="visit_type")
df = pd.concat([df, visit_type_dummies], axis=1)
 
# 4. Define features (X) and target (y)
features = ["rating", "wait_days"] + list(visit_type_dummies.columns)
X = df[features]
y = df["future_no_show_flag"]
 
# 5. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
 
# 6. Train the model
# Using Logistic Regression as it is highly interpretable for clinical triage
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
 
# 7. Evaluate the model
y_pred = model.predict(X_test)
print("--- Model Performance ---")
print(classification_report(y_test, y_pred))
print(f"ROC AUC Score: {roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]):.2f}")
 
# 8. Add probability predictions back to the original dataframe
df["no_show_prob"] = model.predict_proba(X)[:, 1]
 
# Save the trained results
df.to_csv("patient_noshow_predictions.csv", index=False)
print("\nPredictions saved to 'patient_noshow_predictions.csv'")
 
 
# ---------- 10. COST OF AT-RISK NO-SHOWS ----------
# At-Risk patients account for ~79.5% of all no-shows.
# Total: ~$15.9M over 3y; excess above the satisfied baseline: ~$13.3M (~$4.4M/yr)
 
avg_revenue = df.loc[df["has_claim"] & (df["total_claim_amount"] > 0),
                     "total_claim_amount"].mean()
 
ns = df[df["status"] == "No Show"].groupby("at_risk").size()
share_ar = ns[True] / ns.sum()
cost_ar = ns[True] * avg_revenue
 
print(f"No-shows — Satisfied: {ns[False]:,} | At-Risk: {ns[True]:,} "
      f"({share_ar:.1%} of all no-shows)")
print(f"Cost of At-Risk no-shows: ${cost_ar/1e6:.1f}M over 3 years "
      f"(~${cost_ar/3/1e6:.1f}M per year)")
 
# Excess no-shows: what At-Risk patients cost ABOVE the satisfied baseline.
# This isolates the cost of dissatisfaction itself — even satisfied
# patients no-show ~3.8% of the time, so we only count the difference.
g = df.dropna(subset=["future_no_show_flag"]).groupby("at_risk")["future_no_show_flag"]
rate_sat, rate_ar = g.mean()[False], g.mean()[True]
excess = (rate_ar - rate_sat) * g.size()[True]
excess_cost = excess * avg_revenue
print(f"Future no-show rate — Satisfied: {rate_sat:.1%} | At-Risk: {rate_ar:.1%}")
print(f"Excess no-shows attributable to dissatisfaction: {excess:,.0f} "
      f"= ${excess_cost/1e6:.1f}M over 3 years (~${excess_cost/3/1e6:.1f}M per year)")
 
pd.Series({"Satisfied\npatients": ns[False] * avg_revenue / 1e6,
           "At-Risk\npatients": cost_ar / 1e6}).plot(
    kind="bar", color=["#0F5C5C", "#D8492F"], figsize=(6, 4))
plt.title(f"At-Risk patients drive {share_ar:.0%} of no-show losses "
          f"(≈${cost_ar/1e6:.0f}M)")
plt.ylabel("Lost visit revenue ($M)"); plt.xticks(rotation=0)
plt.show()
 
 
# ---------- 11. EVALUATION OF THE THEME-MAPPING MODEL ----------
# Two-tier evaluation:
#   (a) In-dataset: does the model reproduce the existing labels? (100% —
#       but the dataset only has 18 unique canned comments, so this just
#       proves memorization, not understanding.)
#   (b) Held-out: 30 hand-written NOVEL comments the model has never seen.
#       This measures real generalization (~60% accuracy, macro F1 ~0.58).
 
from sklearn.metrics import classification_report, confusion_matrix
 
# Rebuild the theme column if the CSV was reloaded and it got wiped
if "theme" not in df.columns:
    df["theme"] = df["complaint_category"].fillna("Positive / No Complaint")
 
 
# (a) In-dataset accuracy
labeled = df[df["comment"].notna() & (df["comment"].str.strip() != "")]
in_acc = (model_theme.predict(labeled["comment"]) == labeled["theme"]).mean()
print(f"(a) In-dataset accuracy ({len(labeled):,} rows): {in_acc:.1%}")
print("    Caveat: only 18 unique comments exist, so this reflects")
print("    memorization — reliable for labeling THIS dataset only.\n")
 
# (b) Held-out test set of novel comments (never seen in training)
test = pd.DataFrame([
    ("They double charged my credit card for the copay.", "Billing"),
    ("My statement shows fees I was never told about.", "Billing"),
    ("The invoice doesn't match what insurance says I owe.", "Billing"),
    ("I got sent to collections over a bill I already paid.", "Billing"),
    ("Why am I paying out of pocket when I have coverage?", "Billing"),
    ("The doctor interrupted me constantly and seemed annoyed.", "Provider"),
    ("My physician spent less than five minutes with me.", "Provider"),
    ("The specialist was arrogant and wouldn't answer questions.", "Provider"),
    ("The nurse rolled her eyes when I described my pain.", "Provider"),
    ("He didn't even examine me before prescribing pills.", "Provider"),
    ("The exam room smelled bad and the chairs were broken.", "Facilities"),
    ("There was no wheelchair access at the side entrance.", "Facilities"),
    ("The elevator has been out for weeks.", "Facilities"),
    ("The bathroom had no soap and looked filthy.", "Facilities"),
    ("Parking took forever and cost a fortune.", "Facilities"),
    ("I sat in the lobby for three hours before being seen.", "Wait Time"),
    ("My appointment was delayed twice and then started late.", "Wait Time"),
    ("Six weeks to see a specialist is unacceptable.", "Wait Time"),
    ("Still stuck on a waitlist after two months.", "Wait Time"),
    ("They made me wait so long I had to leave for work.", "Wait Time"),
    ("Nobody told me my appointment had been moved.", "Communication"),
    ("I found out my results by accident weeks later.", "Communication"),
    ("The front desk gave me wrong instructions for my fast.", "Communication"),
    ("Emails go unanswered and the phone line just rings.", "Communication"),
    ("No one explained the referral process to me.", "Communication"),
    ("Wonderful staff, felt cared for the whole time.", "Positive / No Complaint"),
    ("Quick, easy, and the doctor was very thorough.", "Positive / No Complaint"),
    ("Best clinic experience I've had in years.", "Positive / No Complaint"),
    ("Check-in was a breeze and everyone was friendly.", "Positive / No Complaint"),
    ("All good, see you at the next checkup.", "Positive / No Complaint"),
], columns=["comment", "theme"])
 
pred = model_theme.predict(test["comment"])
print("(b) Held-out evaluation on 30 novel comments:")
print(classification_report(test["theme"], pred, zero_division=0))
 
labels = sorted(test["theme"].unique())
cm = pd.DataFrame(confusion_matrix(test["theme"], pred, labels=labels),
                  index=labels, columns=labels)
print("Confusion matrix (rows = true, columns = predicted):")
print(cm, "\n")
 
misses = test[pred != test["theme"]].assign(predicted=pred[pred != test["theme"]])
print("Misclassified examples:")
for _, r in misses.iterrows():
    print(f"  '{r['comment']}' -> {r['predicted']} (true: {r['theme']})")
 
# Confusion-matrix heatmap
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, cmap="Reds")
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm.iloc[i, j], ha="center", va="center",
                color="white" if cm.iloc[i, j] > cm.values.max()/2 else "black")
acc = (pred == test["theme"]).mean()
ax.set_title(f"Theme model on novel comments — {acc:.0%} accuracy\n"
             "(100% in-dataset, but the model memorizes 18 canned phrases)")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.show()
 
# Takeaway: reliable for auto-labeling this dataset; NOT production-ready
# for new free text (~60% on unseen phrasing, Wait Time recall only 20%).
# Next step would be a few hundred real labeled comments per theme, or an
# embedding/LLM-based classifier instead of TF-IDF + Naive Bayes.
